In [ ]:
import cloudpickle

import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd

import mat73

import sys
sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic_noprint import NMF_logistic
from sklearn.metrics import roc_auc_score
from scipy.io import savemat

In [ ]:
dir_name = '/home/austin/Aggression2/SingleRegionPredictive/'
fname = dir_name + 'SingleRegion_Aggression_data.mat'
my_dict = mat73.loadmat(fname)

TrainsetMouse = my_dict['TrainsetMouse']
mbeh_all2 = my_dict['mbeh_all2']
mcond_all2 = my_dict['mcond_all2']
mouse_all2 = my_dict['mouse_all2']
mpow_all2 = my_dict['mpow_all2']
mpow_all3s = my_dict['mpow_all3s']
mtimecondbeh2 = my_dict['mtimecondbeh2']
mu_all3 = my_dict['mu_all3']
testsetMouse = my_dict['testsetMouse']


In [ ]:
TrainsetMouse = [['Mouse048'],['Mouse127'],['Mouse128'],['Mouse409'],
                ['Mouse439'],['Mouse442'],['Mouse447'],['Mouse448']]

In [ ]:
N_samples = len(mouse_all2)

mice_all = []
for mouse in TrainsetMouse:
    mice_all.append(mouse[0])

trainset = []
for i in range(len(TrainsetMouse)):
    trainset.append(TrainsetMouse[i][0])

print(mice_all)
train_idxs = np.zeros(N_samples)
mouse_idxs = np.zeros(N_samples)
for i in range(N_samples):
    if mouse_all2[i][0] in trainset:
        train_idxs[i] = 1
        mouse_idxs[i] = trainset.index(mouse_all2[i][0]) + 1

In [ ]:
# Select the points
idxs_pos = (mcond_all2==1)&(mbeh_all2>0)
idx_neg = ((mcond_all2==2)&(mbeh_all2>0))
selection_indices = idxs_pos|idx_neg

y = np.zeros(N_samples)
y[idxs_pos] = 1

# Extract data
mpower_reduced = mpow_all2[:,:,selection_indices]
y_reduced = y[selection_indices]
train_idxs_reduced = train_idxs[selection_indices]
y_train = y_reduced[train_idxs_reduced==1]
mouse_idxs_reduced = mouse_idxs[selection_indices]
m_train = mouse_idxs_reduced[train_idxs_reduced==1]

In [ ]:
# Load models
with open('AggresionTwoPro.p','rb') as f:
    myDict = cloudpickle.load(f)
model_list = myDict['models']

In [ ]:
myDict.keys()

In [ ]:
n_models = len(myDict.keys())

In [ ]:
region_list = []
S_list = []
S_positive_list = []
S_negative_list = []
auc_vals = np.zeros((n_models,8))
count = 0
rlist = ['IL','LHb','LSN','MDThal','MeA','NAc','OFC','PL','V1','VHipp','VMHvl']


for i in range(11):
    for j in range(i+1,11):
        XT1 = np.squeeze(mpower_reduced[:,i,:])
        XT2 = np.squeeze(mpower_reduced[:,j,:])
        X1 = np.transpose(XT1)
        X2 = np.transpose(XT2)
        X = np.hstack((X1,X2))
        X = X*10
        X[X>6] = 6
        
        key = rlist[i] + ' + ' + rlist[j]
        region_list.append(key)
        
        model = myDict[key]
    
        Xtrain = X[train_idxs_reduced==1]
        Xtest = X[train_idxs_reduced==0]
        S_train = model.transform(Xtrain)
        
        S_list.append(S_train)
        S_positive_list.append(S_train[y_train==1])
        S_negative_list.append(S_train[y_train==1])
        for k in range(8):
            my_auc = roc_auc_score(y_train[m_train==1+k],
                    np.squeeze(S_train[m_train==1+k,0]*model.Phi))
            auc_vals[count,k] = my_auc
        count += 1


m_d = {'S_list':S_list,'aucs':auc_vals,'mouse_index':m_train,
       'mouse_label':TrainsetMouse,'idxs_positive':y_train,
      'region_list':region_list,'S_positive':S_positive_list,
      }
    #print(i,auc_vals[i])

In [ ]:
np.savetxt('aucs_test_two.csv',auc_vals,fmt='%0.8f',delimiter=',')

myDict = {'S_list':S_list,'aucs':auc_vals,'mouse_index':m_train,
         'mouse_label':TrainsetMouse,'idxs_positive':y_train,
         'region_list':region_list,'S_positive':S_positive_list,
         'S_negative':S_negative_list}
with open('UrineProjectionTwo.p','wb') as f:
    cloudpickle.dump(myDict,f)

In [ ]:
file_path = "two_region_list.txt"

# Write the list of strings to a text file
with open(file_path, "w") as file:
    for region in region_list:
        file.write(region + "\n")

In [ ]:
savemat('UrineProjection_TwoRegion.mat',myDict)

In [ ]:
np.amax(auc_vals)